# 07 — Model Evaluation & Comparison
Aggregate all model metrics, run the Diebold-Mariano test, and produce thesis-ready charts.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import glob

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

## 1. Load all metrics files

In [ ]:
metrics_files = glob.glob('../data/processed/metrics_*.csv')
all_metrics   = pd.concat([pd.read_csv(f) for f in metrics_files], ignore_index=True)
all_metrics   = all_metrics.sort_values('rmse').reset_index(drop=True)
all_metrics

## 2. Summary table (thesis-ready)

In [ ]:
# Highlight best per column
styled = (
    all_metrics.style
    .highlight_min(subset=['rmse', 'mae'], color='lightgreen')
    .highlight_max(subset=['dir_acc'],     color='lightgreen')
    .format({'rmse': '{:.6f}', 'mae': '{:.6f}', 'dir_acc': '{:.3f}'})
)
styled

## 3. Bar charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric, label in zip(axes,
                              ['rmse', 'mae', 'dir_acc'],
                              ['RMSE (lower better)', 'MAE (lower better)',
                               'Directional Accuracy (higher better)']):
    bars = ax.barh(all_metrics['model'], all_metrics[metric],
                   color=sns.color_palette('tab10', len(all_metrics)))
    ax.set_title(label)
    ax.set_xlabel(metric.upper())
    if metric == 'dir_acc':
        ax.axvline(0.5, color='red', linestyle='--', lw=1, label='Random')
        ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../data/processed/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Diebold-Mariano test
Tests whether two models have statistically different forecast accuracy.
H₀: equal predictive accuracy.


In [ ]:
def diebold_mariano(e1: np.ndarray, e2: np.ndarray, h: int = 1) -> dict:
    """
    e1, e2 : forecast error arrays (actual - predicted)
    h      : forecast horizon
    Returns DM statistic and p-value (two-sided).
    """
    d   = e1**2 - e2**2
    T   = len(d)
    d_bar = d.mean()

    # Newey-West variance with lag h-1
    gamma0 = np.var(d, ddof=1)
    gammas = [np.cov(d[k:], d[:-k])[0, 1] for k in range(1, h)]
    nw_var = gamma0 + 2 * sum(gammas)

    dm_stat = d_bar / np.sqrt(nw_var / T)
    p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    return {'DM stat': round(dm_stat, 4), 'p-value': round(p_value, 4),
            'Reject H₀ (5%)': p_value < 0.05}


# Example usage — populate with actual error arrays from your notebooks:
# errors_arima = actuals - preds_arima
# errors_lstm  = actuals - preds_lstm
# diebold_mariano(errors_arima, errors_lstm)

print('DM test function ready. Load error arrays from notebooks 03–05 to run.')

## 5. Sentiment ablation — does it help?

In [ ]:
# Compare with-sentiment vs without-sentiment models
# Group models by whether they include sentiment
if 'with_sentiment' in all_metrics.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    for has_sent, grp in all_metrics.groupby('with_sentiment'):
        label = 'With sentiment' if has_sent else 'Without sentiment'
        ax.scatter(grp['model'], grp['rmse'], label=label, s=80)
    ax.set_title('RMSE: With vs Without Sentiment')
    ax.set_ylabel('RMSE')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Add a with_sentiment column to your metrics CSVs to enable this chart.')

## 6. Silver Squeeze episode zoom-in
Key test of the retail-sentiment hypothesis: do sentiment models outperform during Feb 2021?


In [ ]:
prices = pd.read_csv('../data/raw/daily_prices.csv', index_col=0, parse_dates=True)
prices.index = prices.index.tz_localize(None)

squeeze = prices['silver']['2021-01-01':'2021-04-30']

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(squeeze, lw=1.5, color='steelblue')
ax.axvspan('2021-01-28', '2021-02-05', alpha=0.2, color='red',
           label='Silver Squeeze (Jan 28 – Feb 5)')
ax.set_title('Silver Price — Squeeze Episode')
ax.set_ylabel('USD/oz')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Export final metrics table (LaTeX)

In [ ]:
latex = all_metrics.to_latex(index=False, float_format='%.6f',
                              caption='Out-of-sample forecast accuracy by model',
                              label='tab:results')
print(latex)

with open('../data/processed/results_table.tex', 'w') as f:
    f.write(latex)
print('Saved LaTeX table.')